# DOOM VLM Agent — Unified Game Notebook

Test VLM agents on any VizDoom scenario: solo challenges or multiplayer deathmatch.

### Game types

| Type | Description |
|------|-------------|
| **Solo** | Classic VizDoom training scenarios — agent plays alone vs built-in enemies. Death ends the episode. |
| **Deathmatch** | 1-4 VLM agents compete against each other and/or bots. Respawn on death. |

### Deathmatch modes

| Mode | Description |
|------|-------------|
| **Benchmark** | Each agent plays solo vs bots (sequentially). Fair comparison — identical conditions. |
| **Arena** | All agents + bots in one game (multiprocessing). Direct PvP — faster model = more actions. |

### How to use

1. **Run All** (Ctrl+Shift+Enter) — hides code, installs deps, shows widgets
2. **Configure agents** — add 1-4 agents with model/API settings
3. **Choose game type** — Solo scenario or Deathmatch map
4. **Click "Run Game"** and watch the live preview

In [ ]:
# Hide all code cells — run this first, then Run All
from IPython.display import HTML, display
display(HTML("""
<style>
/* JupyterLab */
.jp-Cell-inputWrapper { display: none !important; }
/* Classic Notebook */
div.input { display: none !important; }
/* Hide "Click to add a cell" between cells */
.jp-Notebook-cell .jp-cell-placeholder,
.jp-Notebook .jp-cell-placeholder { display: none !important; }
</style>
"""))

In [ ]:
# === Setup: Dependencies, Imports, Constants ===
# This cell installs dependencies and defines all core constants.
# It is collapsed by default — click to expand if needed.

from __future__ import annotations

import os
import platform
import subprocess
import sys

# ── System dependencies (Linux only) ──
if platform.system() == "Linux":
    subprocess.run(
        "apt-get update -qq && apt-get install -y -qq ffmpeg fonts-dejavu-core libsdl2-dev zstd pciutils lshw > /dev/null 2>&1",
        shell=True, check=False,
    )

# ── Python dependencies ──
_pip_result = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "vizdoom", "Pillow", "numpy", "requests", "ipywidgets"],
               check=False, capture_output=True)
if _pip_result.returncode != 0:
    print(f"WARNING: pip install failed (exit {_pip_result.returncode}):")
    print(_pip_result.stderr.decode(errors="replace")[:500])

import shutil
# ── ffmpeg (needed for MP4 recording) ──
if not shutil.which("ffmpeg"):
    if platform.system() == "Linux":
        subprocess.run("apt-get install -y -qq ffmpeg > /dev/null 2>&1",
                       shell=True, check=False)
    elif platform.system() == "Darwin":
        subprocess.run("brew install ffmpeg 2>/dev/null || true",
                       shell=True, check=False)
    if not shutil.which("ffmpeg"):
        print("WARNING: ffmpeg not found — MP4 recording will not work")

# ── Imports (safe now that deps are installed) ──
import base64
import collections
import io
import json
import logging
import multiprocessing
import re
import shutil
import tempfile
import threading
import time
import traceback
from dataclasses import dataclass
from datetime import datetime, timezone
from multiprocessing import Process, Queue, Event as MPEvent
from pathlib import Path
from typing import Any

import numpy as np
import requests
from PIL import Image, ImageDraw, ImageFont
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML as DHTML

import vizdoom as vzd

# WARNING: "fork" from a multithreaded process (Jupyter) can deadlock on macOS.
# Python 3.12+ defaults to "spawn" on macOS for good reason.
# TODO: migrate arena to threads or extract run_dm_loop to a .py module for spawn compat.
try:
    multiprocessing.set_start_method("fork")
except RuntimeError:
    pass

os.environ["SDL_VIDEODRIVER"] = "dummy"

# --- Paths ---
BASE_DIR = Path.cwd()
WORKSPACE_ROOT = BASE_DIR / "workspace"
# Per-run directories — reassigned by _run_game before each run
LOG_DIR = BASE_DIR / "dm_logs"
RESULTS_DIR = BASE_DIR / "dm_results"
SCREENSHOT_DIR = BASE_DIR / "dm_screenshots"

logger = logging.getLogger("doom_dm")

FOV_DEGREES = 90
SEARCH_TURN_DELTA = 3

# Button indices
BTN_TURN   = 0
BTN_ATTACK = 1
BTN_FWD    = 2
BTN_BACK   = 3
BTN_LEFT   = 4
BTN_RIGHT  = 5
BTN_USE    = 6
NUM_BUTTONS = 7

# Font loader (cached)
_font_cache: dict[int, ImageFont.ImageFont] = {}

def _load_font(size: int) -> ImageFont.ImageFont:
    if size in _font_cache:
        return _font_cache[size]
    candidates = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/TTF/DejaVuSans.ttf",
        "/System/Library/Fonts/Helvetica.ttc",
        "/System/Library/Fonts/SFNSMono.ttf",
    ]
    for path in candidates:
        try:
            font = ImageFont.truetype(path, size)
            _font_cache[size] = font
            return font
        except (OSError, IOError):
            continue
    font = ImageFont.load_default()
    _font_cache[size] = font
    return font

# Agent colors (up to 4 agents)
AGENT_COLORS = [
    {"colorset": 0, "css": "#00cc00", "name": "green"},
    {"colorset": 4, "css": "#cc0000", "name": "red"},
    {"colorset": 6, "css": "#0066cc", "name": "blue"},
    {"colorset": 2, "css": "#cccc00", "name": "yellow"},
]

# Default tool descriptions
DEFAULT_SHOOT_DESC = "Shoot ONLY when a monster or soldier is clearly visible on screen. Do not shoot walls."
DEFAULT_MOVE_DESC = "Default action. Explore the map, turn to search, or walk toward items."
DEFAULT_COLUMN_DESC = "Enemy column 1-{grid_cols}"
DEFAULT_DIRECTION_DESC = "Direction to move"

# Default prompt templates
DEFAULT_SYSTEM_PROMPT = "DOOM. Columns 1-{grid_cols} on screen. shoot(column) at enemy, move(direction) to explore."
DEFAULT_USER_PROMPT = "HP={health} AMMO={ammo}"

# ── Scenario catalog ──
SCENARIO_CATALOG = {
    # Solo scenarios (PLAYER mode, KILLCOUNT, no respawn)
    "Basic": {
        "game_type": "solo", "cfg": "basic.cfg",
        "desc": "A single stationary monster. Kill it ASAP. 300 tics.",
    },
    "Simpler Basic": {
        "game_type": "solo", "cfg": "simpler_basic.cfg",
        "desc": "Monster centered on screen. Simplest possible test.",
    },
    "Rocket Basic": {
        "game_type": "solo", "cfg": "rocket_basic.cfg",
        "desc": "Like Basic but with a rocket launcher \u2014 projectile has travel time.",
    },
    "Defend the Center": {
        "game_type": "solo", "cfg": "defend_the_center.cfg",
        "desc": "Monsters approach from all sides. Defend 360\u00b0. 26 bullets.",
    },
    "Defend the Line": {
        "game_type": "solo", "cfg": "defend_the_line.cfg",
        "desc": "Monsters attack from the front. Hold your ground.",
    },
    "Deadly Corridor": {
        "game_type": "solo", "cfg": "deadly_corridor.cfg",
        "desc": "Fight through a corridor of enemies to reach the end. Skill 5/5.",
    },
    "Predict Position": {
        "game_type": "solo", "cfg": "predict_position.cfg",
        "desc": "A monster runs along a wall \u2014 predict its position and fire a rocket.",
    },
    "Health Gathering": {
        "game_type": "solo", "cfg": "health_gathering.cfg",
        "desc": "Toxic floor drains HP. Collect medkits to survive. No enemies.",
    },
    "Health Gathering Supreme": {
        "game_type": "solo", "cfg": "health_gathering_supreme.cfg",
        "desc": "Harder Health Gathering. Floor drains faster, fewer medkits.",
    },
    "My Way Home": {
        "game_type": "solo", "cfg": "my_way_home.cfg",
        "desc": "Navigate a maze to find the exit. No enemies. Pure navigation.",
    },
    "Take Cover": {
        "game_type": "solo", "cfg": "take_cover.cfg",
        "desc": "Dodge incoming fireballs by moving left/right. No weapons. Survival.",
    },
    # Deathmatch scenarios (ASYNC_PLAYER, FRAGCOUNT/DEATHCOUNT, respawn)
    "CIG map01 (small)": {
        "game_type": "dm", "cfg": "cig.cfg", "map": "map01",
        "desc": "Small VizDoom CIG competition arena. Fast-paced combat.",
    },
    "CIG map02 (large)": {
        "game_type": "dm", "cfg": "cig.cfg", "map": "map02",
        "desc": "Large CIG arena. More room to maneuver.",
    },
    "Multi DM": {
        "game_type": "dm", "cfg": "multi.cfg", "map": "map01",
        "desc": "Standard multiplayer deathmatch map.",
    },
    "Deathmatch": {
        "game_type": "dm", "cfg": "deathmatch.cfg", "map": "map01",
        "desc": "Classic Doom II deathmatch. Large map with many weapons.",
    },
}


def get_scenario_meta(name: str) -> dict:
    """Get scenario metadata from catalog."""
    return SCENARIO_CATALOG[name]


def is_solo_scenario(name: str) -> bool:
    return SCENARIO_CATALOG[name]["game_type"] == "solo"


class _SafeDict(dict):
    """dict subclass that returns '{key}' for missing keys — safe prompt formatting."""
    def __missing__(self, key):
        logger.warning("Unknown prompt tag: %s", key)
        return "{" + key + "}"


def format_prompt(template: str, **kwargs) -> str:
    """Format a prompt template, leaving unknown tags as-is."""
    return template.format_map(_SafeDict(**kwargs))


@dataclass
class AgentConfig:
    name: str
    api_url: str
    model: str
    api_key: str = ""
    colorset: int = 0
    color_css: str = "#00cc00"
    temperature: float = 0.7
    top_p: float = 0.8
    top_k: int = 20
    presence_penalty: float = 1.5
    max_tokens: int = 200
    system_prompt: str = DEFAULT_SYSTEM_PROMPT
    user_prompt: str = DEFAULT_USER_PROMPT
    history_len: int = 0
    history_images: bool = False
    shoot_desc: str = DEFAULT_SHOOT_DESC
    move_desc: str = DEFAULT_MOVE_DESC
    column_desc: str = DEFAULT_COLUMN_DESC
    direction_desc: str = DEFAULT_DIRECTION_DESC



In [ ]:
from IPython.display import clear_output as _co
_co(wait=True)

# === Agent Configuration UI ===

_agent_blocks: list[dict[str, Any]] = []
_agents_container = widgets.VBox()
_agents_status = widgets.HTML()

_TAGS_HINT_HTML = (
    "<details><summary style='cursor:pointer; opacity:0.6; font-size:12px'>"
    "Available template tags</summary>"
    "<div style='font-family:monospace; font-size:11px; padding:4px; "
    "background:rgba(128,128,128,0.1); border:1px solid rgba(128,128,128,0.25); "
    "border-radius:4px; margin-top:4px'>"
    "<b>System prompt:</b><br>"
    "&nbsp;&nbsp;<code>{grid_cols}</code> — grid column count (e.g. 5)<br><br>"
    "<b>User prompt (updated each step):</b><br>"
    "&nbsp;&nbsp;<code>{health}</code> — current HP<br>"
    "&nbsp;&nbsp;<code>{ammo}</code> — current ammo<br>"
    "&nbsp;&nbsp;<code>{frags}</code> — frag count (DM)<br>"
    "&nbsp;&nbsp;<code>{deaths}</code> — death count (DM)<br>"
    "&nbsp;&nbsp;<code>{kills}</code> — kill count (Solo)<br>"
    "&nbsp;&nbsp;<code>{reward}</code> — cumulative reward (Solo)<br>"
    "&nbsp;&nbsp;<code>{step}</code> — game step number<br>"
    "&nbsp;&nbsp;<code>{grid_cols}</code> — grid column count<br>"
    "<br><b>Tool &amp; param descriptions:</b><br>"
    "&nbsp;&nbsp;<code>{grid_cols}</code> — grid column count<br>"
    "</div></details>"
)

def _make_agent_block(idx: int) -> dict[str, Any]:
    color = AGENT_COLORS[idx % len(AGENT_COLORS)]
    w_name = widgets.Text(
        value=f"Agent-{idx+1}",
        description="Name:",
        style={"description_width": "80px"},
        layout=widgets.Layout(width="300px"),
    )
    w_api = widgets.Text(
        value="http://localhost:1234/v1/chat/completions",
        description="API URL:",
        style={"description_width": "80px"},
        layout=widgets.Layout(width="500px"),
    )
    w_model = widgets.Text(
        value="qwen3.5-0.8b",
        description="Model:",
        style={"description_width": "80px"},
        layout=widgets.Layout(width="400px"),
    )
    w_api_key = widgets.Password(
        value="",
        description="API Key:",
        placeholder="Leave empty for local (LM Studio)",
        style={"description_width": "80px"},
        layout=widgets.Layout(width="500px"),
    )
    w_system_prompt = widgets.Textarea(
        value=DEFAULT_SYSTEM_PROMPT,
        description="System:",
        style={"description_width": "80px"},
        layout=widgets.Layout(width="100%", height="60px"),
    )
    w_user_prompt = widgets.Textarea(
        value=DEFAULT_USER_PROMPT,
        description="User:",
        style={"description_width": "80px"},
        layout=widgets.Layout(width="100%", height="40px"),
    )
    w_shoot_desc = widgets.Textarea(
        value=DEFAULT_SHOOT_DESC,
        description="shoot():",
        style={"description_width": "80px"},
        layout=widgets.Layout(width="100%", height="40px"),
    )
    w_move_desc = widgets.Textarea(
        value=DEFAULT_MOVE_DESC,
        description="move():",
        style={"description_width": "80px"},
        layout=widgets.Layout(width="100%", height="40px"),
    )
    w_column_desc = widgets.Textarea(
        value=DEFAULT_COLUMN_DESC,
        description="column:",
        style={"description_width": "80px"},
        layout=widgets.Layout(width="100%", height="30px"),
    )
    w_direction_desc = widgets.Textarea(
        value=DEFAULT_DIRECTION_DESC,
        description="direction:",
        style={"description_width": "80px"},
        layout=widgets.Layout(width="100%", height="30px"),
    )
    w_temp = widgets.FloatSlider(
        value=0.7, min=0.0, max=2.0, step=0.1,
        description="Temperature:", style={"description_width": "130px"}, readout_format=".1f",
    )
    w_top_p = widgets.FloatSlider(
        value=0.8, min=0.0, max=1.0, step=0.05,
        description="Top-p:", style={"description_width": "130px"}, readout_format=".2f",
    )
    w_top_k = widgets.IntSlider(
        value=20, min=1, max=100, step=1,
        description="Top-k:", style={"description_width": "130px"},
    )
    w_pres = widgets.FloatSlider(
        value=1.5, min=0.0, max=3.0, step=0.1,
        description="Presence pen.:", style={"description_width": "130px"}, readout_format=".1f",
    )
    w_max_tok = widgets.IntSlider(
        value=200, min=50, max=1000, step=50,
        description="Max tokens:", style={"description_width": "130px"},
    )
    w_history = widgets.IntSlider(
        value=0, min=0, max=10, step=1,
        description="History:", style={"description_width": "130px"},
    )
    w_history_images = widgets.Checkbox(
        value=False, description="Include images in history",
        style={"description_width": "auto"},
        layout=widgets.Layout(display="none"),
    )
    def _on_history_change(change):
        w_history_images.layout.display = "" if change["new"] > 0 else "none"
    w_history.observe(_on_history_change, names="value")

    prompt_accordion = widgets.Accordion(
        children=[widgets.VBox([
            widgets.HTML("<b style='font-size:12px'>System prompt</b> <span style='font-size:11px;color:#888'>— sent once at start, sets agent role</span>"),
            w_system_prompt,
            widgets.HTML("<b style='font-size:12px'>User prompt</b> <span style='font-size:11px;color:#888'>— sent every step with game state</span>"),
            w_user_prompt,
            widgets.HTML("<hr style='margin:6px 0;opacity:0.3'>"),
            widgets.HTML("<b style='font-size:12px'>shoot() description</b> <span style='font-size:11px;color:#888'>— when to call shoot(column)</span>"),
            w_shoot_desc,
            widgets.HTML("<b style='font-size:12px'>move() description</b> <span style='font-size:11px;color:#888'>— when to call move(direction)</span>"),
            w_move_desc,
            widgets.HTML("<hr style='margin:6px 0;opacity:0.3'>"),
            widgets.HTML("<b style='font-size:12px'>column param</b> <span style='font-size:11px;color:#888'>— describes shoot's column argument</span>"),
            w_column_desc,
            widgets.HTML("<b style='font-size:12px'>direction param</b> <span style='font-size:11px;color:#888'>— describes move's direction argument</span>"),
            w_direction_desc,
            widgets.HTML(_TAGS_HINT_HTML),
        ])])
    prompt_accordion.set_title(0, "Prompts")
    prompt_accordion.selected_index = None  # collapsed

    llm_accordion = widgets.Accordion(children=[widgets.VBox([w_temp, w_top_p, w_top_k, w_pres, w_max_tok, w_history, w_history_images])])
    llm_accordion.set_title(0, "LLM Parameters")
    llm_accordion.selected_index = None  # collapsed

    css = color['css']
    cname = color['name']
    color_badge = widgets.HTML(
        value=f"<span style='display:inline-block;width:14px;height:14px;"
              f"background:{css};border-radius:3px;margin-right:6px'"
              f"></span> <b>Agent {idx+1}</b> ({cname})"
    )

    block_box = widgets.VBox(
        [color_badge, w_name, w_api, w_model, w_api_key, prompt_accordion, llm_accordion],
        layout=widgets.Layout(
            border=f"2px solid {color['css']}",
            padding="8px", margin="4px 0",
        ),
    )

    return {
        "box": block_box,
        "w_name": w_name, "w_api": w_api, "w_model": w_model,
        "w_api_key": w_api_key,
        "w_system_prompt": w_system_prompt, "w_user_prompt": w_user_prompt,
        "w_temp": w_temp, "w_top_p": w_top_p, "w_top_k": w_top_k,
        "w_pres": w_pres, "w_max_tok": w_max_tok,
        "w_shoot_desc": w_shoot_desc, "w_move_desc": w_move_desc,
        "w_column_desc": w_column_desc, "w_direction_desc": w_direction_desc,
        "w_history": w_history,
        "w_history_images": w_history_images,
        "color": color,
    }


def _refresh_agent_ui():
    _agents_container.children = [b["box"] for b in _agent_blocks]
    _agents_status.value = f"<i>{len(_agent_blocks)} agent(s) configured</i>"


def _add_agent(btn):
    if len(_agent_blocks) >= 4:
        _agents_status.value = "<b style='color:red'>Max 4 agents</b>"
        return
    _agent_blocks.append(_make_agent_block(len(_agent_blocks)))
    _refresh_agent_ui()


def _remove_agent(btn):
    if len(_agent_blocks) <= 1:
        _agents_status.value = "<b style='color:red'>Need at least 1 agent</b>"
        return
    _agent_blocks.pop()
    _refresh_agent_ui()


def get_agent_configs() -> list[AgentConfig]:
    configs = []
    seen_names: set[str] = set()
    for i, b in enumerate(_agent_blocks):
        color = AGENT_COLORS[i % len(AGENT_COLORS)]
        name = b["w_name"].value.strip()
        if not name:
            import warnings
            warnings.warn(f"Agent {i+1} has empty name, using default")
            name = f"Agent-{i+1}"
        if name in seen_names:
            import warnings
            warnings.warn(f"Duplicate agent name '{name}', appending suffix")
            name = f"{name}_{i}"
        seen_names.add(name)
        api_url = b["w_api"].value.strip()
        if not api_url.startswith(("http://", "https://")):
            import warnings
            warnings.warn(f"Agent {name}: API URL does not start with http(s)://")
        configs.append(AgentConfig(
            name=name,
            api_url=api_url,
            model=b["w_model"].value,
            api_key=b["w_api_key"].value,
            colorset=color["colorset"],
            color_css=color["css"],
            temperature=b["w_temp"].value,
            top_p=b["w_top_p"].value,
            top_k=b["w_top_k"].value,
            presence_penalty=b["w_pres"].value,
            max_tokens=b["w_max_tok"].value,
            system_prompt=b["w_system_prompt"].value,
            user_prompt=b["w_user_prompt"].value,
            shoot_desc=b["w_shoot_desc"].value,
            move_desc=b["w_move_desc"].value,
            column_desc=b["w_column_desc"].value,
            direction_desc=b["w_direction_desc"].value,
            history_len=b["w_history"].value,
            history_images=b["w_history_images"].value,
        ))
    return configs


# Initialize with 1 agent
_agent_blocks.append(_make_agent_block(0))

_add_btn = widgets.Button(description="Add Agent", button_style="info", icon="plus")
_remove_btn = widgets.Button(description="Remove Agent", button_style="warning", icon="minus")
_add_btn.on_click(_add_agent)
_remove_btn.on_click(_remove_agent)

_refresh_agent_ui()
display(widgets.VBox([
    widgets.HBox([_add_btn, _remove_btn, _agents_status]),
    _agents_container,
]))

In [ ]:
# === Game Settings ===

_HINT_STYLE = "opacity:0.6; font-size:12px; margin-left:10px"

w_game_type = widgets.ToggleButtons(
    options=["Solo", "Deathmatch"],
    value="Solo",
    description="Game type:",
    tooltips=[
        "Classic VizDoom scenarios — agent plays alone vs built-in enemies",
        "Multiplayer deathmatch — agents compete vs agents and/or bots",
    ],
    style={"description_width": "100px", "button_width": "120px"},
)

w_scenario = widgets.Dropdown(
    options=[],
    description="Scenario:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="350px"),
)

w_scenario_desc = widgets.HTML(value="")

# ── DM-only widgets ──
w_game_mode = widgets.ToggleButtons(
    options=["Benchmark", "Arena"],
    value="Benchmark",
    description="DM mode:",
    tooltips=[
        "Each agent plays solo vs bots (sequential). Fair comparison.",
        "All agents play together (multiprocessing). Direct PvP.",
    ],
    style={"description_width": "100px"},
)
w_dm_timing = widgets.RadioButtons(
    options=[("Sync (game waits for VLM)", "sync"),
             ("Realtime (faster VLM = more actions)", "realtime")],
    value="realtime",
    description="Timing:",
    style={"description_width": "100px"},
)
w_num_bots = widgets.IntSlider(
    value=4, min=0, max=7, step=1,
    description="Bots:",
    style={"description_width": "120px"},
)
w_timelimit = widgets.FloatSlider(
    value=5.0, min=1.0, max=20.0, step=0.5,
    description="Time limit (min):",
    style={"description_width": "120px"},
    readout_format=".1f",
)

# ── Common widgets ──
w_tics_per_action = widgets.IntSlider(
    value=4, min=1, max=8, step=1,
    description="Tics/action:",
    style={"description_width": "120px"},
)
w_grid_cols = widgets.IntSlider(
    value=9, min=3, max=10, step=1,
    description="Grid columns:",
    style={"description_width": "120px"},
)
w_max_dim = widgets.IntSlider(
    value=1024, min=128, max=1024, step=64,
    description="Image size (px):",
    style={"description_width": "120px"},
)
w_benchmark_episodes = widgets.IntSlider(
    value=3, min=1, max=10, step=1,
    description="Episodes:",
    style={"description_width": "120px"},
)
w_record_fmt = widgets.Dropdown(
    options=["none", "gif", "mp4"],
    value="mp4",
    description="Record:",
    style={"description_width": "120px"},
)


# ── Containers for conditional visibility ──
_dm_only_box = widgets.VBox([
    w_game_mode,
    w_dm_timing,
    widgets.HBox([w_num_bots,
        widgets.HTML(f"<small style='{_HINT_STYLE}'>Number of built-in AI opponents. More bots = more chaotic.</small>")]),
    widgets.HBox([w_timelimit,
        widgets.HTML(f"<small style='{_HINT_STYLE}'>Game duration in minutes. Longer = more frags but slower runs.</small>")]),
])

_episodes_box = widgets.VBox([
    widgets.HBox([w_benchmark_episodes,
        widgets.HTML(f"<small style='{_HINT_STYLE}'>Rounds per agent. More episodes = more reliable statistics.</small>")]),
])


def _update_scenario_desc():
    name = w_scenario.value
    if name and name in SCENARIO_CATALOG:
        meta = SCENARIO_CATALOG[name]
        w_scenario_desc.value = (
            f"<div style='padding:6px 10px; margin:2px 0 6px 0; "
            f"border-left:3px solid #888; background:rgba(128,128,128,0.08); "
            f"font-size:13px'>{meta['desc']}</div>"
        )
    else:
        w_scenario_desc.value = ""


def _on_game_type_change(change):
    is_dm = change["new"] == "Deathmatch"
    _dm_only_box.layout.display = "" if is_dm else "none"
    # Repopulate scenario dropdown
    if is_dm:
        opts = [k for k, v in SCENARIO_CATALOG.items() if v["game_type"] == "dm"]
    else:
        opts = [k for k, v in SCENARIO_CATALOG.items() if v["game_type"] == "solo"]
    w_scenario.options = opts
    if opts:
        w_scenario.value = opts[0]
    _update_scenario_desc()
    # Show episodes box for Solo always, for DM only in Benchmark mode
    if is_dm:
        _episodes_box.layout.display = "" if w_game_mode.value == "Benchmark" else "none"
    else:
        _episodes_box.layout.display = ""


def _on_scenario_change(change):
    _update_scenario_desc()


def _on_dm_mode_change(change):
    _episodes_box.layout.display = "" if change["new"] == "Benchmark" else "none"


w_game_type.observe(_on_game_type_change, names="value")
w_scenario.observe(_on_scenario_change, names="value")
w_game_mode.observe(_on_dm_mode_change, names="value")

# Initialize with Solo selected
_on_game_type_change({"new": "Solo"})

display(widgets.VBox([
    w_game_type,
    w_scenario,
    w_scenario_desc,
    _dm_only_box,
    widgets.HBox([w_tics_per_action,
        widgets.HTML(f"<small style='{_HINT_STYLE}'>Game ticks between VLM decisions. Lower = faster reactions but more API calls. 4 recommended.</small>")]),
    widgets.HBox([w_grid_cols,
        widgets.HTML(f"<small style='{_HINT_STYLE}'>Screen is split into this many aiming zones. More = finer aim but harder for the model.</small>")]),
    widgets.HBox([w_max_dim,
        widgets.HTML(f"<small style='{_HINT_STYLE}'>Screenshot resolution sent to VLM. Larger = more detail but slower and more VRAM.</small>")]),
    _episodes_box,
    widgets.HBox([w_record_fmt,
        widgets.HTML(f"<small style='{_HINT_STYLE}'>Save gameplay video. GIF for quick preview, MP4 for quality (requires ffmpeg).</small>")]),
]))

In [ ]:
# === Image Utilities, VLM Tools & Communication ===
# This cell defines image processing, grid overlay, VLM API calls, and response parsing.
# It is collapsed by default — click to expand if needed.


def screen_to_pil(screen_buffer: np.ndarray) -> Image.Image:
    """Convert a VizDoom screen buffer (CHW or HWC) to a PIL Image."""
    if screen_buffer.ndim == 3 and screen_buffer.shape[0] in (1, 3, 4):
        img_array = np.transpose(screen_buffer, (1, 2, 0))
    else:
        img_array = screen_buffer
    if img_array.ndim == 2:
        return Image.fromarray(img_array, mode="L")
    if img_array.shape[2] == 1:
        return Image.fromarray(img_array[:, :, 0], mode="L")
    return Image.fromarray(img_array)


def draw_grid_overlay(img: Image.Image, grid_cols: int) -> Image.Image:
    """Draw a numbered grid overlay on the image."""
    overlay = img.copy().convert("RGB")
    draw = ImageDraw.Draw(overlay)
    w, h = overlay.size
    col_width = w // grid_cols
    font = _load_font(24)

    for i in range(grid_cols):
        x = i * col_width
        if i > 0:
            draw.line([(x, 0), (x, h)], fill=(255, 255, 0), width=2)
        label = str(i + 1)
        cx = x + col_width // 2
        bbox = draw.textbbox((0, 0), label, font=font)
        tw = bbox[2] - bbox[0]
        for dx in (-1, 0, 1):
            for dy in (-1, 0, 1):
                draw.text((cx - tw // 2 + dx, 5 + dy), label, fill=(0, 0, 0), font=font)
        draw.text((cx - tw // 2, 5), label, fill=(255, 255, 0), font=font)

    return overlay


def _wrap_text(text: str, font: ImageFont.ImageFont, max_width: int) -> list[str]:
    """Word-wrap text to fit within max_width pixels."""
    words = text.split()
    if not words:
        return [text]
    lines: list[str] = []
    current = words[0]
    for word in words[1:]:
        test = current + " " + word
        bbox = font.getbbox(test)
        if bbox[2] - bbox[0] <= max_width:
            current = test
        else:
            lines.append(current)
            current = word
    lines.append(current)
    return lines


def encode_frame(img: Image.Image, max_dim: int = 320) -> str:
    """Resize + JPEG encode + base64."""
    resized = img.copy()
    resized.thumbnail((max_dim, max_dim))
    if resized.mode not in ("RGB", "L"):
        resized = resized.convert("RGB")
    buf = io.BytesIO()
    resized.save(buf, format="JPEG", quality=80)
    return base64.b64encode(buf.getvalue()).decode("utf-8")




def compute_grid_turn_deltas(grid_cols: int, tics_per_action: int = 4) -> dict[int, int]:
    """Compute per-tic turn deltas for each column."""
    deltas = {}
    center = (grid_cols + 1) / 2.0
    col_width_deg = FOV_DEGREES / grid_cols
    for col in range(1, grid_cols + 1):
        offset_deg = (col - center) * col_width_deg
        deltas[col] = round(offset_deg / tics_per_action)
    deltas[0] = SEARCH_TURN_DELTA
    return deltas


def make_vlm_tools(grid_cols: int,
                   shoot_desc: str = DEFAULT_SHOOT_DESC,
                   move_desc: str = DEFAULT_MOVE_DESC,
                   column_desc: str = DEFAULT_COLUMN_DESC,
                   direction_desc: str = DEFAULT_DIRECTION_DESC) -> list[dict]:
    return [
        {
            "type": "function",
            "function": {
                "name": "shoot",
                "description": shoot_desc,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "column": {
                            "type": "integer",
                            "description": format_prompt(column_desc, grid_cols=grid_cols),
                            "enum": list(range(1, grid_cols + 1)),
                        },
                    },
                    "required": ["column"],
                },
            },
        },
        {
            "type": "function",
            "function": {
                "name": "move",
                "description": move_desc,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "direction": {
                            "type": "string",
                            "description": direction_desc,
                            "enum": ["forward", "backward", "left", "right",
                                     "strafe_left", "strafe_right"],
                        },
                    },
                    "required": ["direction"],
                },
            },
        },
    ]


def parse_vlm_response(data: dict, grid_cols: int) -> dict[str, str]:
    """Parse VLM response — extract tool call shoot(column) or move(direction)."""
    message = data["choices"][0]["message"]
    tool_calls = message.get("tool_calls")

    if tool_calls:
        tc = tool_calls[0]
        fn_name = tc["function"]["name"]
        args_raw = tc["function"]["arguments"]
        try:
            args = json.loads(args_raw) if isinstance(args_raw, str) else args_raw
        except json.JSONDecodeError:
            return {"shoot": "no", "cell": "0", "move": "forward",
                    "reason": f"malformed tool args: {str(args_raw)[:100]}"}

        if fn_name == "shoot":
            col = int(args["column"])
            if not 1 <= col <= grid_cols:
                raise ValueError(f"shoot column {col} out of range 1-{grid_cols}")
            return {
                "shoot": "yes", "cell": str(col),
                "move": "none", "reason": f"shoot(column={col})",
            }

        if fn_name == "move":
            direction = args["direction"]
            return {
                "shoot": "no", "cell": "0",
                "move": direction, "reason": f"move(direction={direction})",
            }

        raise ValueError(f"Unknown tool call: {fn_name}")

    # No tool call — strip reasoning tags and special tokens, fallback to forward
    content = (message.get("content") or "").strip()
    content = re.sub(r"<think>.*?</think>", "", content, flags=re.DOTALL).strip()
    content = re.sub(r"<\|im_end\|>|<\|end\|>|<\|eot_id\|>|</s>", "", content).strip()
    return {
        "shoot": "no", "cell": "0",
        "move": "forward",
        "reason": content or "no tool call, moving forward",
    }


def build_action(parsed: dict[str, str], turn_deltas: dict[int, int]) -> list[float]:
    """Translate parsed VLM response into a VizDoom action vector."""
    action = [0.0] * NUM_BUTTONS

    cell = int(parsed["cell"])
    shoot = parsed["shoot"] == "yes"
    move = parsed["move"]
    search_delta = float(turn_deltas[0])

    if shoot:
        action[BTN_TURN] = float(turn_deltas[cell])
        action[BTN_ATTACK] = 1.0
    elif move != "none":
        if move == "forward":
            action[BTN_FWD] = 1.0
        elif move == "backward":
            action[BTN_BACK] = 1.0
        elif move == "left":
            action[BTN_TURN] = -search_delta
        elif move == "right":
            action[BTN_TURN] = search_delta
        elif move == "strafe_left":
            action[BTN_LEFT] = 1.0
        elif move == "strafe_right":
            action[BTN_RIGHT] = 1.0
    else:
        action[BTN_TURN] = search_delta

    return action


def call_vlm(
    b64_image: str,
    user_text: str,
    system_prompt: str,
    tools: list[dict],
    *,
    api_url: str,
    model: str,
    temperature: float = 0.7,
    top_p: float = 0.8,
    top_k: int = 20,
    presence_penalty: float = 1.5,
    max_tokens: int = 200,
    history: list[dict] | None = None,
    timeout: float = 120,
    api_key: str = "",
) -> tuple[dict, float]:
    """Send screenshot with grid overlay to VLM. All params are explicit."""
    messages = [{"role": "system", "content": system_prompt}]
    if history:
        messages.extend(history)
    messages.append({
        "role": "user",
        "content": [
            {"type": "text", "text": user_text},
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{b64_image}"},
            },
        ],
    })
    payload = {
        "model": model,
        "messages": messages,
        "tools": tools,
        "tool_choice": "required",
        "max_tokens": max_tokens,
        "temperature": temperature,
        "top_p": top_p,
        "top_k": top_k,
        "presence_penalty": presence_penalty,
    }

    headers = {"Content-Type": "application/json"}
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"

    t0 = time.perf_counter()
    for attempt in range(3):
        try:
            resp = requests.post(api_url, json=payload, headers=headers, timeout=timeout)
            if resp.status_code >= 400:
                body = resp.text[:500]
                logger.warning("[%s] VLM attempt %d HTTP %d: %s",
                               model, attempt + 1, resp.status_code, body)
                if attempt < 2:
                    time.sleep(1 * (attempt + 1))
                    continue
                else:
                    logger.error("[%s] VLM failed after 3 attempts: HTTP %d", model, resp.status_code)
                    raise requests.exceptions.HTTPError(
                        f"HTTP {resp.status_code}: {body}", response=resp)
            data = resp.json()
            elapsed = time.perf_counter() - t0
            # Log VLM response
            msg = data.get("choices", [{}])[0].get("message", {})
            tool_calls = msg.get("tool_calls")
            if tool_calls:
                tc = tool_calls[0]
                logger.info("[%s] VLM %.1fs tool_call: %s(%s)",
                            model, elapsed, tc["function"]["name"],
                            tc["function"]["arguments"])
            else:
                content = (msg.get("content") or "")[:200]
                if not content:
                    logger.warning("[%s] VLM %.1fs empty response (no tool_calls, no content)",
                                   model, elapsed)
                else:
                    logger.info("[%s] VLM %.1fs content: %s", model, elapsed, content)
            return data, elapsed
        except requests.exceptions.RequestException as e:
            if attempt < 2:
                wait = 1 * (attempt + 1)
                logger.warning("[%s] VLM attempt %d failed: %s, retrying in %ds",
                               model, attempt + 1, e, wait)
                time.sleep(wait)
            else:
                logger.error("[%s] VLM failed after 3 attempts: %s", model, e)
                raise




def make_dm_system_prompt(grid_cols: int, template: str = DEFAULT_SYSTEM_PROMPT) -> str:
    return format_prompt(template, grid_cols=grid_cols)


def save_debug_screenshot(
    img_with_grid: Image.Image,
    player_name: str,
    episode: int,
    step: int,
    parsed: dict[str, str],
    raw_response: str,
    action_desc: str,
    reward: float,
    health: float,
    ammo: float,
    latency: float,
) -> None:
    """Save a debug image with game screenshot and VLM response."""
    SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)

    game_img = img_with_grid.copy().convert("RGB")
    gw, gh = game_img.size
    font = _load_font(14)

    padding = 8
    line_height = 18
    max_text_width = gw - padding * 2

    header = f"{player_name} | Episode {episode}  Step {step}  |  HP={health:.0f}  AMMO={ammo:.0f}  |  Latency={latency:.1f}s"
    raw_lines = []
    for rl in raw_response.split("\n"):
        raw_lines.extend(_wrap_text(rl, font, max_text_width) if rl.strip() else [""])
    action_line = f"Game: {action_desc}"

    all_lines = (
        _wrap_text(header, font, max_text_width)
        + [""]
        + raw_lines
        + [""]
        + _wrap_text(action_line, font, max_text_width)
    )

    text_height = len(all_lines) * line_height + padding * 2
    combined = Image.new("RGB", (gw, gh + text_height), (30, 30, 30))
    combined.paste(game_img, (0, 0))

    draw = ImageDraw.Draw(combined)
    y = gh + padding
    for line in all_lines:
        draw.text((padding, y), line, fill=(220, 220, 220), font=font)
        y += line_height

    path = SCREENSHOT_DIR / f"{player_name}_ep{episode:02d}_step{step:03d}.png"
    combined.save(path)

In [ ]:
# === Game Engine: Recorder, Setup, Game Loop ===
# This cell defines EpisodeRecorder, game setup functions, and the deathmatch game loop.
# It is collapsed by default — click to expand if needed.


class EpisodeRecorder:
    """Records per-tic frames with overlay, assembles GIF or MP4."""

    def __init__(self, episode: int, scenario: str, fmt: str, grid_cols: int,
                 player_name: str = "agent", fps: int = 12,
                 game_type: str = "dm"):
        self.episode = episode
        self.scenario = scenario
        self.fmt = fmt
        self.grid_cols = grid_cols
        self.player_name = player_name
        self.fps = fps
        self.game_type = game_type
        self._frames: list[Image.Image] = []
        self._tmp_dir: Path | None = None
        self._frame_count = 0
        self._step_ctx: dict[str, Any] = {}
        self._prev_health: float = 100.0

        if fmt == "mp4":
            self._tmp_dir = Path(tempfile.mkdtemp(prefix="doom_rec_"))

    def set_step_context(
        self, step: int, health: float, ammo: float, frags: float,
        parsed: dict[str, str], action_desc: str, reward: float, latency: float,
    ) -> None:
        hp_dropped = health < self._prev_health
        self._prev_health = health
        self._step_ctx = {
            "step": step, "health": health, "ammo": ammo, "frags": frags,
            "reason": parsed.get("reason", ""), "action_desc": action_desc,
            "reward": reward, "latency": latency,
            "hp_dropped": hp_dropped, "dead": health <= 0,
        }

    def capture_tic(self, screen_buffer: np.ndarray, tic_idx: int, total_tics: int) -> None:
        img = screen_to_pil(screen_buffer)
        img = draw_grid_overlay(img, self.grid_cols)
        img = self._draw_overlay(img, tic_idx, total_tics)
        img = img.convert("RGB")

        if self.fmt == "gif":
            self._frames.append(img.quantize(colors=256, method=Image.Quantize.MEDIANCUT))
        else:
            path = self._tmp_dir / f"frame_{self._frame_count:05d}.png"
            img.save(path)

        self._frame_count += 1

    def _draw_overlay(self, img: Image.Image, tic_idx: int, total_tics: int) -> Image.Image:
        img = img.copy().convert("RGBA")
        overlay = Image.new("RGBA", img.size, (0, 0, 0, 0))
        draw = ImageDraw.Draw(overlay)
        w, h = img.size
        ctx = self._step_ctx
        font = _load_font(13)
        font_sm = _load_font(11)

        # Top bar
        draw.rectangle([(0, 30), (w, 56)], fill=(0, 0, 0, 160))
        step = ctx.get("step", 0)
        health = ctx.get("health", 0)
        ammo = ctx.get("ammo", 0)
        frags = ctx.get("frags", 0)
        hp_color = (255, 80, 80) if ctx.get("hp_dropped") else (80, 255, 80)
        top_text = f"Step {step} | Tic {tic_idx + 1}/{total_tics} | "
        draw.text((6, 34), top_text, fill=(255, 255, 255), font=font)
        hp_x = 6 + font.getbbox(top_text)[2]
        hp_str = f"HP={health:.0f}"
        draw.text((hp_x, 34), hp_str, fill=hp_color, font=font)
        ammo_x = hp_x + font.getbbox(hp_str)[2] + 8
        ammo_str = f"AMMO={ammo:.0f}"
        draw.text((ammo_x, 34), ammo_str, fill=(255, 255, 255), font=font)
        frags_x = ammo_x + font.getbbox(ammo_str)[2] + 8
        frag_label = "KILLS" if self.game_type == "solo" else "FRAGS"
        draw.text((frags_x, 34), f"{frag_label}={frags:.0f}", fill=(255, 200, 0), font=font)

        # Bottom bar
        draw.rectangle([(0, h - 24), (w, h)], fill=(0, 0, 0, 160))
        reason = ctx.get("reason", "")
        action_desc = ctx.get("action_desc", "")
        latency = ctx.get("latency", 0)
        bot_text = f"VLM: {reason} | {action_desc} | {latency:.1f}s"
        max_chars = w // 6
        if len(bot_text) > max_chars:
            bot_text = bot_text[:max_chars - 1] + "\u2026"
        draw.text((6, h - 22), bot_text, fill=(200, 200, 200), font=font_sm)

        img = Image.alpha_composite(img, overlay)
        result = img.convert("RGB")
        draw_rgb = ImageDraw.Draw(result)

        if tic_idx == 0:
            for i in range(3):
                draw_rgb.rectangle([(i, i), (w - 1 - i, h - 1 - i)], outline=(0, 255, 0))

        if ctx.get("dead"):
            red_overlay = Image.new("RGB", result.size, (180, 0, 0))
            result = Image.blend(result, red_overlay, alpha=0.4)

        return result

    def finalize(self) -> Path | None:
        if self._frame_count == 0:
            return None
        RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        name = f"{self.player_name}_episode_{self.episode}_{self.scenario}"
        if self.fmt == "gif":
            return self._finalize_gif(name)
        return self._finalize_mp4(name)

    def _finalize_gif(self, name: str) -> Path:
        out = RESULTS_DIR / f"{name}.gif"
        duration_ms = 1000 // self.fps
        self._frames[0].save(
            out, save_all=True, append_images=self._frames[1:],
            duration=duration_ms, loop=0, optimize=False,
        )
        size_mb = out.stat().st_size / 1_000_000
        if size_mb > 50:
            tmp = Path(tempfile.mkdtemp(prefix="doom_gif2mp4_"))
            for i, f in enumerate(self._frames):
                f.convert("RGB").save(tmp / f"frame_{i:05d}.png")
            self._tmp_dir = tmp
            self._finalize_mp4(name)
            out.unlink(missing_ok=True)
            return RESULTS_DIR / f"{name}.mp4"
        return out

    def _finalize_mp4(self, name: str) -> Path | None:
        out = RESULTS_DIR / f"{name}.mp4"
        cmd = [
            "ffmpeg", "-y", "-framerate", str(self.fps),
            "-i", str(self._tmp_dir / "frame_%05d.png"),
            "-c:v", "libx264", "-pix_fmt", "yuv420p",
            "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2",
            str(out),
        ]
        try:
            subprocess.run(cmd, check=True, capture_output=True)
        except FileNotFoundError:
            logger.error("ffmpeg not found — install ffmpeg for MP4 recording")
            return None
        except subprocess.CalledProcessError as e:
            logger.error("ffmpeg failed: %s", e.stderr.decode(errors="replace"))
            return None
        finally:
            if self._tmp_dir and self._tmp_dir.exists():
                shutil.rmtree(self._tmp_dir)
        return out




def _configure_dm_buttons(game: vzd.DoomGame) -> None:
    """Standard button set for deathmatch."""
    game.clear_available_buttons()
    game.add_available_button(vzd.Button.TURN_LEFT_RIGHT_DELTA, 10)
    game.add_available_button(vzd.Button.ATTACK)
    game.add_available_button(vzd.Button.MOVE_FORWARD)
    game.add_available_button(vzd.Button.MOVE_BACKWARD)
    game.add_available_button(vzd.Button.MOVE_LEFT)
    game.add_available_button(vzd.Button.MOVE_RIGHT)
    game.add_available_button(vzd.Button.USE)


def _configure_dm_variables(game: vzd.DoomGame) -> None:
    """Game variables for deathmatch: HEALTH, AMMO2, FRAGCOUNT, DEATHCOUNT."""
    game.clear_available_game_variables()
    game.add_available_game_variable(vzd.GameVariable.HEALTH)
    game.add_available_game_variable(vzd.GameVariable.AMMO2)
    game.add_available_game_variable(vzd.GameVariable.FRAGCOUNT)
    game.add_available_game_variable(vzd.GameVariable.DEATHCOUNT)


def setup_dm_host(
    scenario: str, map_name: str, num_players: int,
    timelimit: float, agent_name: str, colorset: int,
    dm_mode: str = "realtime",
) -> vzd.DoomGame:
    """Set up host for multiplayer deathmatch."""
    agent_name = re.sub(r"[^a-zA-Z0-9_-]", "", agent_name)[:20]
    cfg_path = os.path.join(vzd.scenarios_path, scenario)
    os.environ["SDL_VIDEODRIVER"] = "dummy"

    game = vzd.DoomGame()
    game.load_config(cfg_path)
    game.set_window_visible(False)

    _configure_dm_buttons(game)
    _configure_dm_variables(game)

    game.add_game_args(f"-host {num_players} -deathmatch")
    game.add_game_args(f"+timelimit {timelimit}")
    game.add_game_args("+sv_forcerespawn 1")
    game.add_game_args("+sv_noautoaim 1")
    game.add_game_args("+sv_respawnprotect 1")
    game.add_game_args("+sv_spawnfarthest 1")
    game.add_game_args("+sv_nocrouch 1")
    game.add_game_args(f"+name {agent_name}")
    game.add_game_args(f"+colorset {colorset}")
    game.add_game_args(f"+map {map_name}")

    game.set_mode(vzd.Mode.PLAYER if dm_mode == "sync" else vzd.Mode.ASYNC_PLAYER)
    game.init()
    return game


def setup_dm_join(
    scenario: str, agent_name: str, colorset: int,
    host_address: str = "127.0.0.1",
    dm_mode: str = "realtime",
) -> vzd.DoomGame:
    """Set up client to join a multiplayer deathmatch."""
    agent_name = re.sub(r"[^a-zA-Z0-9_-]", "", agent_name)[:20]
    cfg_path = os.path.join(vzd.scenarios_path, scenario)
    os.environ["SDL_VIDEODRIVER"] = "dummy"

    game = vzd.DoomGame()
    game.load_config(cfg_path)
    game.set_window_visible(False)

    _configure_dm_buttons(game)
    _configure_dm_variables(game)

    game.add_game_args(f"-join {host_address}")
    game.add_game_args(f"+name {agent_name}")
    game.add_game_args(f"+colorset {colorset}")

    game.set_mode(vzd.Mode.PLAYER if dm_mode == "sync" else vzd.Mode.ASYNC_PLAYER)
    game.init()
    return game


def setup_benchmark_game(
    scenario: str, map_name: str, timelimit: float,
    num_bots: int, agent_name: str, colorset: int,
) -> vzd.DoomGame:
    """Set up a single-player deathmatch vs bots (benchmark mode)."""
    agent_name = re.sub(r"[^a-zA-Z0-9_-]", "", agent_name)[:20]
    cfg_path = os.path.join(vzd.scenarios_path, scenario)
    os.environ["SDL_VIDEODRIVER"] = "dummy"

    game = vzd.DoomGame()
    game.load_config(cfg_path)
    game.set_window_visible(False)

    _configure_dm_buttons(game)
    _configure_dm_variables(game)

    game.add_game_args(f"-host 1 -deathmatch")
    game.add_game_args(f"+timelimit {timelimit}")
    game.add_game_args("+sv_forcerespawn 1")
    game.add_game_args("+sv_noautoaim 1")
    game.add_game_args("+sv_respawnprotect 1")
    game.add_game_args("+sv_spawnfarthest 1")
    game.add_game_args("+sv_nocrouch 1")
    game.add_game_args(f"+name {agent_name}")
    game.add_game_args(f"+colorset {colorset}")
    game.add_game_args(f"+map {map_name}")

    game.set_mode(vzd.Mode.PLAYER)
    game.init()

    for _ in range(num_bots):
        game.send_game_command("addbot")

    return game


def get_dm_game_vars(state) -> dict[str, float]:
    """Extract health, ammo, frags, deaths from game state."""
    gv = state.game_variables if state.game_variables is not None else []
    names = ["health", "ammo", "frags", "deaths"]
    if len(gv) < len(names):
        logger.warning("Expected %d game vars, got %d", len(names), len(gv))
    result: dict[str, float] = {}
    for name, val in zip(names, gv):
        result[name] = float(val)
    return result




def run_dm_loop(
    agent_cfg: dict,
    game_settings: dict,
    status_queue: Any,
    stop_event: Any,
    is_host: bool = True,
    host_address: str = "127.0.0.1",
) -> None:
    """
    Main deathmatch loop for a single agent.
    Runs in a subprocess (arena) or thread (benchmark).
    agent_cfg and game_settings are plain dicts (for pickling).
    """
    import os
    os.environ["SDL_VIDEODRIVER"] = "dummy"

    agent_name = agent_cfg["name"]
    episode = game_settings.get("episode", 1)
    game = None
    recorder = None

    try:
        mode = game_settings["mode"]
        scenario = game_settings["scenario"]
        map_name = game_settings["map_name"]
        timelimit = game_settings["timelimit"]
        num_bots = game_settings.get("num_bots", 0)
        num_players = game_settings.get("num_players", 1)
        tics_per_action = game_settings["tics_per_action"]
        grid_cols = game_settings["grid_cols"]
        max_dim = game_settings["max_dim"]
        record_fmt = game_settings.get("record_fmt")
        dm_mode = game_settings.get("dm_mode", "realtime")
        include_images = agent_cfg.get("history_images", False)

        # Create game
        if mode == "benchmark":
            game = setup_benchmark_game(
                scenario, map_name, timelimit,
                num_bots, agent_name, agent_cfg["colorset"],
            )
        elif is_host:
            game = setup_dm_host(
                scenario, map_name, num_players,
                timelimit, agent_name, agent_cfg["colorset"],
                dm_mode=dm_mode,
            )
            # Host adds bots after init
            for _ in range(num_bots):
                game.send_game_command("addbot")
        else:
            game = setup_dm_join(
                scenario, agent_name, agent_cfg["colorset"],
                host_address=host_address,
                dm_mode=dm_mode,
            )

        turn_deltas = compute_grid_turn_deltas(grid_cols, tics_per_action)
        system_prompt = make_dm_system_prompt(
            grid_cols,
            agent_cfg.get("system_prompt", DEFAULT_SYSTEM_PROMPT),
        )
        user_prompt_template = agent_cfg.get("user_prompt", DEFAULT_USER_PROMPT)
        tools = make_vlm_tools(
            grid_cols,
            shoot_desc=agent_cfg.get("shoot_desc", DEFAULT_SHOOT_DESC),
            move_desc=agent_cfg.get("move_desc", DEFAULT_MOVE_DESC),
            column_desc=agent_cfg.get("column_desc", DEFAULT_COLUMN_DESC),
            direction_desc=agent_cfg.get("direction_desc", DEFAULT_DIRECTION_DESC),
        )

        # Conversation history buffer
        history_len = agent_cfg.get("history_len", 0)
        history_buffer = collections.deque(maxlen=history_len) if history_len > 0 else None

        recorder = (
            EpisodeRecorder(
                episode, scenario.replace(".cfg", ""), record_fmt,
                grid_cols, player_name=agent_name, game_type="dm",
            )
            if record_fmt
            else None
        )

        game.new_episode()
        step = 0
        total_latency = 0.0
        respawn_count = 0
        frags = 0.0
        deaths = 0.0

        logger.info("[%s] Episode %d started", agent_name, episode)

        status_queue.put({
            "type": "started",
            "agent": agent_name,
            "episode": episode,
        })

        while not game.is_episode_finished():
            if stop_event.is_set():
                break

            # Respawn if dead
            if game.is_player_dead():
                game.respawn_player()
                respawn_count += 1
                continue

            state = game.get_state()
            if state is None:
                game.make_action([0.0] * NUM_BUTTONS, 1)
                continue

            step += 1
            gv = get_dm_game_vars(state)
            health = gv.get("health", 100.0)
            ammo = gv.get("ammo", 0.0)
            frags = gv.get("frags", 0.0)
            deaths = gv.get("deaths", 0.0)

            img = screen_to_pil(state.screen_buffer)
            img_with_grid = draw_grid_overlay(img, grid_cols)
            b64 = encode_frame(img_with_grid, max_dim=max_dim)

            # Format user prompt from template
            user_text = format_prompt(
                user_prompt_template,
                health=int(health), ammo=int(ammo),
                frags=int(frags), deaths=int(deaths),
                step=step, grid_cols=grid_cols,
            )

            # Build history from buffer
            history = [msg for turn in history_buffer for msg in turn] if history_buffer else None

            try:
                vlm_data, latency = call_vlm(
                    b64, user_text, system_prompt, tools,
                    api_url=agent_cfg["api_url"],
                    model=agent_cfg["model"],
                    temperature=agent_cfg["temperature"],
                    top_p=agent_cfg["top_p"],
                    top_k=agent_cfg["top_k"],
                    presence_penalty=agent_cfg["presence_penalty"],
                    max_tokens=agent_cfg["max_tokens"],
                    history=history,
                    api_key=agent_cfg.get("api_key", ""),
                )
                parsed = parse_vlm_response(vlm_data, grid_cols)

                # Save turn to history buffer on success
                if history_buffer is not None:
                    if include_images:
                        user_entry = {"role": "user", "content": [
                            {"type": "text", "text": user_text},
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
                        ]}
                    else:
                        user_entry = {"role": "user", "content": user_text}
                    assistant_msg = vlm_data["choices"][0]["message"]
                    asst_entry = {"role": "assistant", "content": assistant_msg.get("content")}
                    if assistant_msg.get("tool_calls"):
                        asst_entry["tool_calls"] = assistant_msg["tool_calls"]
                    turn = [user_entry, asst_entry]
                    if assistant_msg.get("tool_calls"):
                        for tc in assistant_msg["tool_calls"]:
                            turn.append({"role": "tool", "tool_call_id": tc["id"], "content": "ok"})
                    history_buffer.append(turn)

            except Exception as e:
                logger.warning("[%s] step %d VLM error: %s", agent_name, step, e)
                parsed = {"shoot": "no", "cell": "0", "move": "forward", "reason": f"VLM error: {e}"}
                vlm_data = {}
                latency = 0.0

            action_vec = build_action(parsed, turn_deltas)
            total_latency += latency

            # Action description
            cell = int(parsed.get("cell", "0"))
            shoot = parsed.get("shoot", "no") == "yes"
            move_cmd = parsed.get("move", "none")
            act_str = f"SHOOT@{cell}" if shoot else (f"MOVE:{move_cmd}" if move_cmd != "none" else "search")
            turn_delta = action_vec[BTN_TURN]
            action_desc = f"turn={turn_delta:+.0f}\u00b0 act={act_str}"

            logger.debug("[%s] step %d: %s | HP=%.0f AMMO=%.0f F=%.0f D=%.0f",
                         agent_name, step, action_desc, health, ammo, frags, deaths)

            if recorder:
                recorder.set_step_context(
                    step, health, ammo, frags,
                    parsed, action_desc, 0.0, latency,
                )

            # Execute action
            reward = 0.0
            for tic in range(tics_per_action):
                if game.is_episode_finished():
                    break
                if game.is_player_dead():
                    break
                if mode == "benchmark":
                    tic_reward = game.make_action(action_vec, 1)
                    reward += tic_reward
                else:
                    game.make_action(action_vec, 1)

                if recorder and not game.is_episode_finished():
                    tic_state = game.get_state()
                    if tic_state:
                        recorder.capture_tic(tic_state.screen_buffer, tic, tics_per_action)

            # Get updated frags/deaths
            if not game.is_episode_finished() and not game.is_player_dead():
                post_state = game.get_state()
                if post_state:
                    post_gv = get_dm_game_vars(post_state)
                    frags = post_gv.get("frags", frags)
                    deaths = post_gv.get("deaths", deaths)

            # Send frame preview to display
            frame_b64 = encode_frame(img_with_grid, max_dim=380)
            status_queue.put({
                "type": "step",
                "agent": agent_name,
                "episode": episode,
                "step": step,
                "frags": frags,
                "deaths": deaths,
                "health": health,
                "ammo": ammo,
                "latency": round(latency, 2),
                "action": act_str,
                "frame_b64": frame_b64,
            })

            save_debug_screenshot(
                img_with_grid, agent_name, episode, step,
                parsed, parsed["reason"], action_desc,
                reward, health, ammo, latency,
            )

        # Episode finished — read final frag/death counts
        try:
            final_frags = game.get_game_variable(vzd.GameVariable.FRAGCOUNT)
            final_deaths = game.get_game_variable(vzd.GameVariable.DEATHCOUNT)
        except Exception:
            final_frags = frags
            final_deaths = deaths
        scoreboard = []
        try:
            server = game.get_server_state()
            for i in range(len(server.players_in_game)):
                if server.players_in_game[i]:
                    scoreboard.append({
                        "name": server.players_names[i],
                        "frags": server.players_frags[i],
                    })
        except Exception as e:
            logger.debug("Scoreboard error: %s", e)

        rec_path = None
        if recorder:
            rec_path = recorder.finalize()

        logger.info("[%s] Episode %d done: frags=%.0f deaths=%.0f steps=%d avg_lat=%.1fs",
                    agent_name, episode, final_frags, final_deaths, step,
                    total_latency / max(step, 1))

        status_queue.put({
            "type": "done",
            "agent": agent_name,
            "episode": episode,
            "frags": final_frags,
            "deaths": final_deaths,
            "steps": step,
            "avg_latency": round(total_latency / max(step, 1), 2),
            "scoreboard": scoreboard,
            "recording": str(rec_path) if rec_path else None,
        })

    except Exception as e:
        status_queue.put({
            "type": "error",
            "agent": agent_name,
            "episode": episode,
            "error": f"{type(e).__name__}: {e}",
            "traceback": traceback.format_exc(),
        })
    finally:
        if recorder:
            try:
                recorder.finalize()
            except Exception:
                pass
        if game is not None:
            try:
                game.close()
            except Exception:
                pass


def _configure_solo_variables(game: vzd.DoomGame) -> None:
    """Game variables for solo: HEALTH, AMMO2, KILLCOUNT."""
    game.clear_available_game_variables()
    game.add_available_game_variable(vzd.GameVariable.HEALTH)
    game.add_available_game_variable(vzd.GameVariable.AMMO2)
    game.add_available_game_variable(vzd.GameVariable.KILLCOUNT)


def setup_solo_game(cfg_name: str) -> vzd.DoomGame:
    """Set up a single-player solo scenario."""
    cfg_path = os.path.join(vzd.scenarios_path, cfg_name)
    os.environ["SDL_VIDEODRIVER"] = "dummy"

    game = vzd.DoomGame()
    game.load_config(cfg_path)
    game.set_window_visible(False)
    game.set_mode(vzd.Mode.PLAYER)

    _configure_dm_buttons(game)
    _configure_solo_variables(game)

    game.init()
    return game


def get_solo_game_vars(state) -> dict[str, float]:
    """Extract health, ammo, kills from solo game state."""
    gv = state.game_variables if state.game_variables is not None else []
    names = ["health", "ammo", "kills"]
    if len(gv) < len(names):
        logger.warning("Expected %d game vars, got %d", len(names), len(gv))
    result: dict[str, float] = {}
    for name, val in zip(names, gv):
        result[name] = float(val)
    return result


def run_solo_loop(
    agent_cfg: dict,
    game_settings: dict,
    status_queue: Any,
    stop_event: Any,
) -> None:
    """
    Solo scenario game loop for a single agent.
    Runs in a thread. Death = episode over (no respawn).
    """
    import os
    os.environ["SDL_VIDEODRIVER"] = "dummy"

    agent_name = agent_cfg["name"]
    episode = game_settings.get("episode", 1)
    game = None
    recorder = None

    try:
        cfg_name = game_settings["cfg"]
        tics_per_action = game_settings["tics_per_action"]
        grid_cols = game_settings["grid_cols"]
        max_dim = game_settings["max_dim"]
        record_fmt = game_settings.get("record_fmt")
        scenario_label = game_settings.get("scenario_label", cfg_name)
        include_images = agent_cfg.get("history_images", False)

        game = setup_solo_game(cfg_name)

        turn_deltas = compute_grid_turn_deltas(grid_cols, tics_per_action)
        system_prompt = make_dm_system_prompt(
            grid_cols,
            agent_cfg.get("system_prompt", DEFAULT_SYSTEM_PROMPT),
        )
        user_prompt_template = agent_cfg.get("user_prompt", DEFAULT_USER_PROMPT)
        tools = make_vlm_tools(
            grid_cols,
            shoot_desc=agent_cfg.get("shoot_desc", DEFAULT_SHOOT_DESC),
            move_desc=agent_cfg.get("move_desc", DEFAULT_MOVE_DESC),
            column_desc=agent_cfg.get("column_desc", DEFAULT_COLUMN_DESC),
            direction_desc=agent_cfg.get("direction_desc", DEFAULT_DIRECTION_DESC),
        )

        # Conversation history buffer
        history_len = agent_cfg.get("history_len", 0)
        history_buffer = collections.deque(maxlen=history_len) if history_len > 0 else None

        recorder = (
            EpisodeRecorder(
                episode, scenario_label.replace(" ", "_").lower(), record_fmt,
                grid_cols, player_name=agent_name, game_type="solo",
            )
            if record_fmt
            else None
        )

        game.new_episode()
        step = 0
        total_latency = 0.0
        total_reward = 0.0
        kills = 0
        prev_kills = 0.0

        logger.info("[%s] Solo episode %d started — %s", agent_name, episode, scenario_label)

        status_queue.put({
            "type": "started",
            "agent": agent_name,
            "episode": episode,
        })

        while not game.is_episode_finished():
            if stop_event.is_set():
                break

            # Solo: death = episode over (no respawn)
            if game.is_player_dead():
                break

            state = game.get_state()
            if state is None:
                game.make_action([0.0] * NUM_BUTTONS, 1)
                continue

            step += 1
            gv = get_solo_game_vars(state)
            health = gv.get("health", 100.0)
            ammo = gv.get("ammo", 0.0)
            cur_kills = gv.get("kills", 0.0)

            img = screen_to_pil(state.screen_buffer)
            img_with_grid = draw_grid_overlay(img, grid_cols)
            b64 = encode_frame(img_with_grid, max_dim=max_dim)

            user_text = format_prompt(
                user_prompt_template,
                health=int(health), ammo=int(ammo),
                kills=int(cur_kills), reward=round(total_reward, 1),
                step=step, grid_cols=grid_cols,
                frags=int(cur_kills), deaths=0,
            )

            # Build history from buffer
            history = [msg for turn in history_buffer for msg in turn] if history_buffer else None

            try:
                vlm_data, latency = call_vlm(
                    b64, user_text, system_prompt, tools,
                    api_url=agent_cfg["api_url"],
                    model=agent_cfg["model"],
                    temperature=agent_cfg["temperature"],
                    top_p=agent_cfg["top_p"],
                    top_k=agent_cfg["top_k"],
                    presence_penalty=agent_cfg["presence_penalty"],
                    max_tokens=agent_cfg["max_tokens"],
                    history=history,
                    api_key=agent_cfg.get("api_key", ""),
                )
                parsed = parse_vlm_response(vlm_data, grid_cols)

                # Save turn to history buffer on success
                if history_buffer is not None:
                    if include_images:
                        user_entry = {"role": "user", "content": [
                            {"type": "text", "text": user_text},
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
                        ]}
                    else:
                        user_entry = {"role": "user", "content": user_text}
                    assistant_msg = vlm_data["choices"][0]["message"]
                    asst_entry = {"role": "assistant", "content": assistant_msg.get("content")}
                    if assistant_msg.get("tool_calls"):
                        asst_entry["tool_calls"] = assistant_msg["tool_calls"]
                    turn = [user_entry, asst_entry]
                    if assistant_msg.get("tool_calls"):
                        for tc in assistant_msg["tool_calls"]:
                            turn.append({"role": "tool", "tool_call_id": tc["id"], "content": "ok"})
                    history_buffer.append(turn)

            except Exception as e:
                logger.warning("[%s] step %d VLM error: %s", agent_name, step, e)
                parsed = {"shoot": "no", "cell": "0", "move": "forward",
                          "reason": f"VLM error: {e}"}
                vlm_data = {}
                latency = 0.0

            action_vec = build_action(parsed, turn_deltas)
            total_latency += latency

            cell = int(parsed.get("cell", "0"))
            shoot = parsed.get("shoot", "no") == "yes"
            move_cmd = parsed.get("move", "none")
            act_str = (f"SHOOT@{cell}" if shoot
                       else (f"MOVE:{move_cmd}" if move_cmd != "none" else "search"))
            turn_delta = action_vec[BTN_TURN]
            action_desc = f"turn={turn_delta:+.0f}\u00b0 act={act_str}"

            if recorder:
                recorder.set_step_context(
                    step, health, ammo, cur_kills,
                    parsed, action_desc, total_reward, latency,
                )

            # Execute action tick-by-tick
            reward = 0.0
            for tic in range(tics_per_action):
                if game.is_episode_finished() or game.is_player_dead():
                    break
                tic_reward = game.make_action(action_vec, 1)
                reward += tic_reward
                if recorder and not game.is_episode_finished():
                    tic_state = game.get_state()
                    if tic_state:
                        recorder.capture_tic(tic_state.screen_buffer, tic, tics_per_action)

            total_reward += reward

            # Check kills after action
            if not game.is_episode_finished() and not game.is_player_dead():
                post_state = game.get_state()
                if post_state:
                    post_gv = get_solo_game_vars(post_state)
                    post_kills = post_gv.get("kills", cur_kills)
                    if post_kills > prev_kills:
                        kills += int(post_kills - prev_kills)
                    prev_kills = post_kills

            # Send preview to display
            frame_b64 = encode_frame(img_with_grid, max_dim=380)
            status_queue.put({
                "type": "step",
                "agent": agent_name,
                "episode": episode,
                "step": step,
                "kills": kills,
                "reward": round(total_reward, 1),
                "health": health,
                "ammo": ammo,
                "latency": round(latency, 2),
                "action": act_str,
                "frame_b64": frame_b64,
            })

            save_debug_screenshot(
                img_with_grid, agent_name, episode, step,
                parsed, parsed["reason"], action_desc,
                reward, health, ammo, latency,
            )

        # Episode finished — final kill count from game variable
        try:
            final_kills = game.get_game_variable(vzd.GameVariable.KILLCOUNT)
            if final_kills > prev_kills:
                kills += int(final_kills - prev_kills)
        except Exception:
            pass

        ep_total_reward = game.get_total_reward()

        rec_path = None
        if recorder:
            rec_path = recorder.finalize()

        logger.info("[%s] Solo ep %d done: kills=%d reward=%.1f steps=%d avg_lat=%.1fs",
                    agent_name, episode, kills, ep_total_reward, step,
                    total_latency / max(step, 1))

        status_queue.put({
            "type": "done",
            "agent": agent_name,
            "episode": episode,
            "kills": kills,
            "reward": round(ep_total_reward, 1),
            "steps": step,
            "avg_latency": round(total_latency / max(step, 1), 2),
            "recording": str(rec_path) if rec_path else None,
        })

    except Exception as e:
        status_queue.put({
            "type": "error",
            "agent": agent_name,
            "episode": game_settings.get("episode", 1),
            "error": f"{type(e).__name__}: {e}",
            "traceback": traceback.format_exc(),
        })
    finally:
        if recorder:
            try:
                recorder.finalize()
            except Exception:
                pass
        if game is not None:
            try:
                game.close()
            except Exception:
                pass

In [ ]:
# === Display & Runners: GameDisplay, Benchmark, Arena, Solo Benchmark ===
# This cell defines the live display widget, benchmark runner, arena runner,
# and solo benchmark runner.  It is collapsed by default — click to expand if needed.


class GameDisplay:
    """Multi-agent live display with per-agent frames and scoreboard.

    Parameters
    ----------
    agent_names : list[str]
        Display names for each agent.
    agent_colors : list[str]
        CSS colours matching *agent_names*.
    game_type : str
        ``"dm"`` for deathmatch / arena, ``"solo"`` for solo scenarios.
    """

    def __init__(self, agent_names: list[str], agent_colors: list[str], game_type: str = "dm"):
        self._agent_names = agent_names
        self._agent_colors = {n: c for n, c in zip(agent_names, agent_colors)}
        self._game_type = game_type

        # Per-agent HTML widgets (thread-safe .value updates via comm)
        self._agent_frames: dict[str, widgets.HTML] = {}
        self._agent_boxes: list[widgets.VBox] = []
        for name, color in zip(agent_names, agent_colors):
            frame = widgets.HTML(
                value="<i>waiting...</i>",
                layout=widgets.Layout(height="260px", width="260px"),
            )
            self._agent_frames[name] = frame
            box = widgets.VBox(
                [widgets.HTML(f"<b style='color:{color}'>{name}</b>"), frame],
                layout=widgets.Layout(border=f"2px solid {color}", padding="4px", margin="2px"),
            )
            self._agent_boxes.append(box)

        self._frames_row = widgets.HBox(
            self._agent_boxes,
            layout=widgets.Layout(flex_flow="row wrap"),
        )

        # Scoreboard
        self._scoreboard = widgets.HTML(value="<i>Waiting for data...</i>")

        # Log (HTML widget — thread-safe .value assignment)
        self._log_html = widgets.HTML(
            value="",
            layout=widgets.Layout(height="150px", overflow_y="auto"),
        )
        self._log_lines: list[str] = []

        self._status = {}

        title = "Live Solo View" if game_type == "solo" else "Live Deathmatch View"

        self._box = widgets.VBox([
            widgets.HTML(f"<h3>{title}</h3>"),
            self._frames_row,
            widgets.HTML("<h4>Scoreboard</h4>"),
            self._scoreboard,
            widgets.HTML("<h4>Log</h4>"),
            self._log_html,
        ])

    def show(self):
        display(self._box)

    def update_agent(self, name: str, step_data: dict) -> None:
        """Update the agent's frame preview and track status."""
        self._status[name] = step_data

        if name in self._agent_frames:
            frame_b64 = step_data.get("frame_b64", "")
            step = step_data.get("step", 0)

            if self._game_type == "solo":
                kills = step_data.get("kills", 0)
                reward = step_data.get("reward", 0)
                subtitle = f"Step {step} | K:{kills} R:{reward}"
            else:
                frags = step_data.get("frags", 0)
                deaths = step_data.get("deaths", 0)
                subtitle = f"Step {step} | F:{frags:.0f} D:{deaths:.0f}"

            if frame_b64:
                self._agent_frames[name].value = (
                    f"<img src='data:image/jpeg;base64,{frame_b64}' "
                    f"style='max-width:250px; max-height:250px'/>"
                    f"<br/><small>{subtitle}</small>"
                )

        self._update_scoreboard()

    # ------------------------------------------------------------------
    # Scoreboard dispatching
    # ------------------------------------------------------------------

    def _update_scoreboard(self) -> None:
        if self._game_type == "solo":
            self._update_solo_scoreboard()
        else:
            self._update_dm_scoreboard()

    def _update_solo_scoreboard(self) -> None:
        rows = ""
        for name in self._agent_names:
            s = self._status.get(name, {})
            color = self._agent_colors.get(name, "#ccc")
            kills = s.get("kills", 0)
            reward = s.get("reward", 0)
            health = s.get("health", 0)
            ammo = s.get("ammo", 0)
            lat = s.get("latency", 0)
            action = s.get("action", "-")
            rows += (
                f"<tr>"
                f"<td style='color:{color};font-weight:bold'>{name}</td>"
                f"<td>{kills}</td><td>{reward}</td>"
                f"<td>{health:.0f}</td><td>{ammo:.0f}</td><td>{lat:.1f}s</td><td>{action}</td>"
                f"</tr>"
            )

        self._scoreboard.value = (
            "<table style='border-collapse:collapse; font-family:monospace; width:100%'>"
            "<tr style='background:#333; color:#fff'>"
            "<th style='padding:4px 8px'>Agent</th>"
            "<th style='padding:4px 8px'>Kills</th>"
            "<th style='padding:4px 8px'>Reward</th>"
            "<th style='padding:4px 8px'>HP</th>"
            "<th style='padding:4px 8px'>Ammo</th>"
            "<th style='padding:4px 8px'>Latency</th>"
            "<th style='padding:4px 8px'>Last Action</th>"
            "</tr>"
            + rows +
            "</table>"
        )

    def _update_dm_scoreboard(self) -> None:
        rows = ""
        for name in self._agent_names:
            s = self._status.get(name, {})
            color = self._agent_colors.get(name, "#ccc")
            frags = s.get("frags", 0)
            deaths = s.get("deaths", 0)
            health = s.get("health", 0)
            ammo = s.get("ammo", 0)
            lat = s.get("latency", 0)
            action = s.get("action", "-")
            kd = frags / max(deaths, 1)
            rows += (
                f"<tr>"
                f"<td style='color:{color};font-weight:bold'>{name}</td>"
                f"<td>{frags:.0f}</td><td>{deaths:.0f}</td><td>{kd:.2f}</td>"
                f"<td>{health:.0f}</td><td>{ammo:.0f}</td><td>{lat:.1f}s</td><td>{action}</td>"
                f"</tr>"
            )

        self._scoreboard.value = (
            "<table style='border-collapse:collapse; font-family:monospace; width:100%'>"
            "<tr style='background:#333; color:#fff'>"
            "<th style='padding:4px 8px'>Agent</th>"
            "<th style='padding:4px 8px'>Frags</th>"
            "<th style='padding:4px 8px'>Deaths</th>"
            "<th style='padding:4px 8px'>K/D</th>"
            "<th style='padding:4px 8px'>HP</th>"
            "<th style='padding:4px 8px'>Ammo</th>"
            "<th style='padding:4px 8px'>Latency</th>"
            "<th style='padding:4px 8px'>Last Action</th>"
            "</tr>"
            + rows +
            "</table>"
        )

    def log(self, msg: str) -> None:
        self._log_lines.append(msg)
        # Keep last 50 lines
        if len(self._log_lines) > 50:
            self._log_lines = self._log_lines[-50:]
        self._log_html.value = (
            "<pre style='font-size:12px; margin:0; white-space:pre-wrap'>"
            + "\n".join(self._log_lines)
            + "</pre>"
        )




def run_benchmark(
    agent_configs: list[dict],
    game_settings: dict,
    display_mgr: GameDisplay,
    stop_event: threading.Event,
) -> list[dict]:
    """
    Benchmark mode: each agent plays N episodes sequentially vs bots.
    Returns list of per-agent results.
    """
    import queue

    all_results = []
    num_episodes = game_settings.get("benchmark_episodes", 3)

    for agent_cfg in agent_configs:
        agent_name = agent_cfg["name"]
        display_mgr.log(f"--- Starting benchmark for {agent_name} ({agent_cfg['model']}) ---")
        agent_episodes = []

        for ep in range(1, num_episodes + 1):
            if stop_event.is_set():
                display_mgr.log("Stop requested.")
                break

            display_mgr.log(f"{agent_name}: Episode {ep}/{num_episodes}")
            ep_settings = {**game_settings, "episode": ep}

            status_q = queue.Queue()
            stop_mp = threading.Event()

            # Run game loop in a thread
            t = threading.Thread(
                target=run_dm_loop,
                args=(agent_cfg, ep_settings, status_q, stop_mp),
                kwargs={"is_host": True},
                daemon=True,
            )
            t.start()

            # Poll queue for updates
            ep_result = None
            while t.is_alive() or not status_q.empty():
                if stop_event.is_set():
                    stop_mp.set()

                try:
                    msg = status_q.get(timeout=0.5)
                except queue.Empty:
                    continue

                if msg["type"] == "step":
                    display_mgr.update_agent(agent_name, msg)
                elif msg["type"] == "done":
                    ep_result = msg
                    display_mgr.log(
                        f"{agent_name} ep{ep}: {msg['frags']:.0f} frags, "
                        f"{msg['deaths']:.0f} deaths, {msg['steps']} steps, "
                        f"{msg['avg_latency']:.1f}s avg"
                    )
                elif msg["type"] == "error":
                    display_mgr.log(f"ERROR ({agent_name}): {msg['error']}")
                    ep_result = {"frags": 0, "deaths": 0, "avg_latency": 0, "steps": 0, "error": msg["error"]}

            t.join(timeout=10)
            if t.is_alive():
                logger.warning("[%s] Episode thread still alive, forcing stop", agent_name)
                stop_mp.set()
                t.join(timeout=5)
            if ep_result:
                agent_episodes.append(ep_result)

        all_results.append({
            "agent": agent_name,
            "model": agent_cfg["model"],
            "episodes": agent_episodes,
        })

    return all_results




def run_arena(
    agent_configs: list[dict],
    game_settings: dict,
    display_mgr: GameDisplay,
    stop_event: threading.Event,
) -> list[dict]:
    """
    Arena mode: all agents in one game via multiprocessing.
    Returns list of per-agent results.
    """
    import queue
    num_agents = len(agent_configs)
    arena_settings = {
        **game_settings,
        "mode": "arena",
        "num_players": num_agents,  # only VLM agents count for -host N
        "episode": 1,
    }

    status_q = Queue()  # multiprocessing.Queue
    mp_stop = MPEvent()  # multiprocessing.Event

    processes: list[Process] = []
    for i, agent_cfg in enumerate(agent_configs):
        is_host = (i == 0)
        p = Process(
            target=run_dm_loop,
            args=(agent_cfg, arena_settings, status_q, mp_stop),
            kwargs={"is_host": is_host, "host_address": "127.0.0.1"},
            daemon=True,
        )
        processes.append(p)

    # Start host first, then clients with delay
    display_mgr.log(f"Starting arena with {num_agents} agents + {game_settings.get('num_bots', 0)} bots")
    processes[0].start()
    display_mgr.log(f"Host ({agent_configs[0]['name']}) started")

    for i in range(1, num_agents):
        time.sleep(5)  # delay for clients to join
        processes[i].start()
        display_mgr.log(f"Client ({agent_configs[i]['name']}) started")

    # Poll queue
    results = {}
    done_count = 0
    arena_start = time.time()
    arena_timeout = game_settings.get("timelimit", 5) * 60 + 180

    while done_count < num_agents:
        if time.time() - arena_start > arena_timeout:
            display_mgr.log("Arena timeout — aborting")
            mp_stop.set()
            break

        if stop_event.is_set():
            mp_stop.set()

        # Check if all processes died
        all_dead = all(not p.is_alive() for p in processes)

        try:
            msg = status_q.get(timeout=1.0)
        except queue.Empty:
            if all_dead:
                break
            continue

        if msg["type"] == "step":
            display_mgr.update_agent(msg["agent"], msg)
        elif msg["type"] == "done":
            done_count += 1
            results[msg["agent"]] = msg
            display_mgr.log(
                f"{msg['agent']} finished: {msg['frags']:.0f} frags, "
                f"{msg['deaths']:.0f} deaths"
            )
        elif msg["type"] == "error":
            done_count += 1
            results[msg["agent"]] = msg
            display_mgr.log(f"ERROR ({msg['agent']}): {msg['error']}")
        elif msg["type"] == "started":
            display_mgr.log(f"{msg['agent']} joined the game")

    # Cleanup
    for p in processes:
        p.join(timeout=10)
        if p.is_alive():
            p.terminate()

    # Cleanup multiprocessing queue
    try:
        status_q.close()
        status_q.join_thread()
    except Exception:
        pass

    return [{"agent": name, "result": results.get(name, {})} for name in [c["name"] for c in agent_configs]]




def run_solo_benchmark(
    agent_configs: list[dict],
    game_settings: dict,
    display_mgr: GameDisplay,
    stop_event: threading.Event,
) -> list[dict]:
    """Solo benchmark: each agent plays N episodes of a solo scenario sequentially."""
    import queue

    all_results = []
    num_episodes = game_settings.get("benchmark_episodes", 3)

    for agent_cfg in agent_configs:
        agent_name = agent_cfg["name"]
        display_mgr.log(f"--- Starting solo benchmark for {agent_name} ({agent_cfg['model']}) ---")
        agent_episodes = []

        for ep in range(1, num_episodes + 1):
            if stop_event.is_set():
                display_mgr.log("Stop requested.")
                break

            display_mgr.log(f"{agent_name}: Episode {ep}/{num_episodes}")
            ep_settings = {**game_settings, "episode": ep}

            status_q = queue.Queue()
            stop_th = threading.Event()

            t = threading.Thread(
                target=run_solo_loop,
                args=(agent_cfg, ep_settings, status_q, stop_th),
                daemon=True,
            )
            t.start()

            ep_result = None
            while t.is_alive() or not status_q.empty():
                if stop_event.is_set():
                    stop_th.set()
                try:
                    msg = status_q.get(timeout=0.5)
                except queue.Empty:
                    continue

                if msg["type"] == "step":
                    display_mgr.update_agent(agent_name, msg)
                elif msg["type"] == "done":
                    ep_result = msg
                    display_mgr.log(
                        f"{agent_name} ep{ep}: {msg.get('kills', 0)} kills, "
                        f"reward={msg.get('reward', 0):.1f}, "
                        f"{msg['steps']} steps, {msg['avg_latency']:.1f}s avg"
                    )
                elif msg["type"] == "error":
                    display_mgr.log(f"ERROR ({agent_name}): {msg['error']}")
                    ep_result = {"kills": 0, "reward": 0, "avg_latency": 0,
                                 "steps": 0, "error": msg["error"]}

            t.join(timeout=10)
            if t.is_alive():
                logger.warning("[%s] Episode thread still alive, forcing stop", agent_name)
                stop_th.set()
                t.join(timeout=5)
            if ep_result:
                agent_episodes.append(ep_result)

        all_results.append({
            "agent": agent_name,
            "model": agent_cfg["model"],
            "episodes": agent_episodes,
        })

    return all_results

In [ ]:
# === Run/Stop buttons + main orchestrator ===

_dm_results: list[dict] = []
_dm_mode_used: str = ""
_game_type_used: str = ""
_dm_stop_event = threading.Event()
_dm_output = widgets.VBox()

_run_btn = widgets.Button(description="Run Game", button_style="success", icon="play")
_stop_btn = widgets.Button(description="Stop", button_style="danger", icon="stop")
_dm_status_html = widgets.HTML(value="<i>Ready</i>")

# Display buttons AND the output container at cell init (main thread)
display(widgets.HBox([_run_btn, _stop_btn, _dm_status_html]))
display(_dm_output)


def _on_dm_stop(btn):
    _dm_stop_event.set()
    _dm_status_html.value = "<b style='color:orange'>Stopping...</b>"


def _show_recordings(run_dir: Path):
    """Show download buttons for recordings using Jupyter FileLink."""
    from IPython.display import FileLink
    results_dir = run_dir / "results"
    if not results_dir.exists():
        return
    videos = sorted(results_dir.glob("*.mp4")) + sorted(results_dir.glob("*.gif"))
    if not videos:
        return
    link_widgets = [widgets.HTML("<b style='font-size:14px'>Recordings:</b>")]
    for v in videos:
        size_mb = v.stat().st_size / 1_000_000
        # Use Jupyter files API for download
        rel = v.resolve().relative_to(Path.cwd().resolve())
        url = f"/files/{rel}?_xsrf=&download=1"
        link_widgets.append(widgets.HTML(
            f"<a href='{url}' download='{v.name}' target='_blank' "
            f"style='margin:2px 4px;display:inline-block;"
            f"padding:6px 14px;background:#2a6;color:#fff;border-radius:4px;"
            f"text-decoration:none;font-size:13px'>"
            f"⬇ {v.name} ({size_mb:.1f} MB)</a>"
        ))
    box = widgets.VBox(link_widgets,
        layout=widgets.Layout(padding='8px', border='1px solid #444',
                              border_radius='6px', margin='8px 0'))
    _dm_output.children = list(_dm_output.children) + [box]



def _run_game():
    global _dm_results, _dm_mode_used, _game_type_used
    _dm_results = []
    try:
        _run_game_inner()
    except Exception as e:
        _dm_status_html.value = f"<b style='color:red'>Error: {e}</b>"
        logger.exception("Game run failed")
    finally:
        _run_btn.disabled = False


def _run_game_inner():
    global _dm_results, _dm_mode_used, _game_type_used

    game_type = w_game_type.value  # "Solo" or "Deathmatch"
    _game_type_used = game_type
    scenario_name = w_scenario.value
    scenario_meta = SCENARIO_CATALOG[scenario_name]

    # Common settings
    tics_per_action = w_tics_per_action.value
    grid_cols = w_grid_cols.value
    max_dim = w_max_dim.value
    benchmark_episodes = w_benchmark_episodes.value
    record_fmt = w_record_fmt.value if w_record_fmt.value != "none" else None

    agent_configs = get_agent_configs()
    if not agent_configs:
        _dm_status_html.value = "<b style='color:red'>No agents configured!</b>"
        _run_btn.disabled = False
        return

    # Create per-run workspace: workspace/NNNN_YYYYMMDD_HHMMSS/
    global LOG_DIR, RESULTS_DIR, SCREENSHOT_DIR
    WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
    existing = sorted(d.name for d in WORKSPACE_ROOT.iterdir() if d.is_dir())
    seq = int(existing[-1].split("_")[0]) + 1 if existing else 1
    run_id = f"{seq:04d}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    run_dir = WORKSPACE_ROOT / run_id
    LOG_DIR = run_dir
    RESULTS_DIR = run_dir / "results"
    SCREENSHOT_DIR = run_dir / "screenshots"
    for d in (RESULTS_DIR, SCREENSHOT_DIR):
        d.mkdir(parents=True, exist_ok=True)
    _dm_status_html.value = f"<b style='color:blue'>Workspace: {run_id}</b>"

    # Setup logging
    log_level = logging.DEBUG
    logger.setLevel(log_level)
    for h in logger.handlers[:]:
        h.close()
        logger.removeHandler(h)
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    ch = logging.StreamHandler()
    ch.setLevel(log_level)
    ch.setFormatter(fmt)
    logger.addHandler(ch)
    fh = logging.FileHandler(run_dir / "game.log", mode="w")
    fh.setLevel(log_level)
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    # Log run configuration
    logger.info("=" * 60)
    logger.info("Run: %s", run_id)
    logger.info("Game type: %s | Scenario: %s (%s)", game_type, scenario_name, scenario_meta["cfg"])
    logger.info("Grid: %d cols | Image: %dpx | Tics/action: %d", grid_cols, max_dim, tics_per_action)
    if record_fmt:
        logger.info("Recording: %s", record_fmt)
    for i, ac in enumerate(agent_configs):
        logger.info("--- Agent %d: %s ---", i + 1, ac.name)
        logger.info("  Model: %s @ %s", ac.model, ac.api_url)
        logger.info("  Temp=%.1f top_p=%.2f top_k=%d pres_pen=%.1f max_tok=%d",
                     ac.temperature, ac.top_p, ac.top_k, ac.presence_penalty, ac.max_tokens)
        logger.info("  History: %d turns (images=%s)", ac.history_len, ac.history_images)
        logger.info("  System: %s", ac.system_prompt[:120])
        logger.info("  User:   %s", ac.user_prompt[:120])
    logger.info("=" * 60)

    # Convert AgentConfigs to dicts (for pickling in arena mode)
    agent_dicts = []
    for ac in agent_configs:
        agent_dicts.append({
            "name": ac.name,
            "api_url": ac.api_url,
            "api_key": ac.api_key,
            "model": ac.model,
            "colorset": ac.colorset,
            "color_css": ac.color_css,
            "temperature": ac.temperature,
            "top_p": ac.top_p,
            "top_k": ac.top_k,
            "presence_penalty": ac.presence_penalty,
            "max_tokens": ac.max_tokens,
            "system_prompt": ac.system_prompt,
            "user_prompt": ac.user_prompt,
            "shoot_desc": ac.shoot_desc,
            "move_desc": ac.move_desc,
            "column_desc": ac.column_desc,
            "direction_desc": ac.direction_desc,
            "history_len": ac.history_len,
            "history_images": ac.history_images,
        })

    agent_names = [a["name"] for a in agent_dicts]
    agent_css_colors = [a["color_css"] for a in agent_dicts]

    if game_type == "Solo":
        # Solo game settings
        game_settings = {
            "cfg": scenario_meta["cfg"],
            "scenario_label": scenario_name,
            "tics_per_action": tics_per_action,
            "grid_cols": grid_cols,
            "max_dim": max_dim,
            "benchmark_episodes": benchmark_episodes,
            "record_fmt": record_fmt,
        }

        display_mgr = GameDisplay(agent_names, agent_css_colors, game_type="solo")
        _dm_status_html.value = f"<b style='color:blue'>Starting Solo: {scenario_name}...</b>"
        _dm_output.children = [display_mgr._box]

        display_mgr.log(f"Game type: Solo")
        display_mgr.log(f"Scenario: {scenario_name} ({scenario_meta['cfg']})")
        display_mgr.log(f"Episodes: {benchmark_episodes}")
        agents_str = ", ".join(f"{a['name']} ({a['model']})" for a in agent_dicts)
        display_mgr.log(f"Agents: {agents_str}")

        try:
            _dm_results = run_solo_benchmark(
                agent_dicts, game_settings, display_mgr, _dm_stop_event,
            )
            _dm_status_html.value = "<b style='color:green'>Done!</b>"
            display_mgr.log("=== Solo benchmark complete ===")
            _show_recordings(run_dir)
            # Log summary
            for r in _dm_results:
                for ep in r.get("episodes", []):
                    logger.info("RESULT %s ep%d: kills=%s reward=%s steps=%s avg_lat=%.1fs",
                                r["agent"], ep.get("episode", 0), ep.get("kills", "?"),
                                ep.get("reward", "?"), ep.get("steps", "?"), ep.get("avg_latency", 0))
        except Exception as e:
            _dm_status_html.value = f"<b style='color:red'>Error: {e}</b>"
            display_mgr.log(f"FATAL: {e}")
            logger.exception("Solo run failed")

    else:
        # Deathmatch
        game_mode = w_game_mode.value
        _dm_mode_used = game_mode
        num_bots = w_num_bots.value
        timelimit = w_timelimit.value

        game_settings = {
            "mode": "benchmark" if game_mode == "Benchmark" else "arena",
            "dm_mode": w_dm_timing.value,
            "scenario": scenario_meta["cfg"],
            "map_name": scenario_meta.get("map", "map01"),
            "num_bots": num_bots,
            "timelimit": timelimit,
            "tics_per_action": tics_per_action,
            "grid_cols": grid_cols,
            "max_dim": max_dim,
            "benchmark_episodes": benchmark_episodes,
            "record_fmt": record_fmt,
        }

        display_mgr = GameDisplay(agent_names, agent_css_colors, game_type="dm")
        _dm_status_html.value = f"<b style='color:blue'>Starting {game_mode}...</b>"
        _dm_output.children = [display_mgr._box]

        display_mgr.log(f"Mode: {game_mode}")
        display_mgr.log(f"Scenario: {scenario_name} ({scenario_meta['cfg']} / {scenario_meta.get('map', 'map01')})")
        agents_str = ", ".join(f"{a['name']} ({a['model']})" for a in agent_dicts)
        display_mgr.log(f"Agents: {agents_str}")
        timing_label = "Sync" if w_dm_timing.value == "sync" else "Realtime"
        display_mgr.log(f"Bots: {num_bots}, Time limit: {timelimit} min, Timing: {timing_label}")

        try:
            if game_mode == "Benchmark":
                _dm_results = run_benchmark(agent_dicts, game_settings, display_mgr, _dm_stop_event)
            else:
                _dm_results = run_arena(agent_dicts, game_settings, display_mgr, _dm_stop_event)

            _dm_status_html.value = "<b style='color:green'>Done!</b>"
            display_mgr.log("=== Deathmatch complete ===")
            _show_recordings(run_dir)
            for r in _dm_results:
                if "episodes" in r:
                    for ep in r["episodes"]:
                        logger.info("RESULT %s ep%d: frags=%s deaths=%s steps=%s avg_lat=%.1fs",
                                    r["agent"], ep.get("episode", 0), ep.get("frags", "?"),
                                    ep.get("deaths", "?"), ep.get("steps", "?"), ep.get("avg_latency", 0))
                elif "result" in r:
                    res = r["result"]
                    logger.info("RESULT %s: frags=%s deaths=%s",
                                r["agent"], res.get("frags", "?"), res.get("deaths", "?"))

        except Exception as e:
            _dm_status_html.value = f"<b style='color:red'>Error: {e}</b>"
            display_mgr.log(f"FATAL: {e}")
            logger.exception("Deathmatch failed")



def _on_dm_run(btn):
    _dm_stop_event.clear()
    _run_btn.disabled = True
    _dm_output.children = []      # clear previous run
    thread = threading.Thread(target=_run_game, daemon=True)
    thread.start()


_run_btn.on_click(_on_dm_run)
_stop_btn.on_click(_on_dm_stop)

## Results

In [ ]:
# === Results table ===

if not _dm_results:
    display(DHTML("<p><i>No results yet. Run a game first.</i></p>"))
elif _game_type_used == "Solo":
    # Solo results: per-agent table with kills + reward
    for agent_result in _dm_results:
        agent_name = agent_result["agent"]
        model = agent_result["model"]
        episodes = agent_result.get("episodes", [])

        rows = ""
        for ep in episodes:
            kills = ep.get("kills", 0)
            reward = ep.get("reward", 0)
            steps = ep.get("steps", 0)
            lat = ep.get("avg_latency", 0)
            rows += (
                f"<tr><td>{ep.get('episode', '?')}</td>"
                f"<td>{kills}</td><td>{reward:.1f}</td>"
                f"<td>{steps}</td><td>{lat:.1f}s</td></tr>\n"
            )

        if episodes:
            n = len(episodes)
            avg_kills = sum(e.get("kills", 0) for e in episodes) / n
            avg_reward = sum(e.get("reward", 0) for e in episodes) / n
            avg_steps = sum(e.get("steps", 0) for e in episodes) / n
            avg_lat = sum(e.get("avg_latency", 0) for e in episodes) / n
            rows += (
                f"<tr style='font-weight:bold; border-top:2px solid #888'>"
                f"<td>AVG</td><td>{avg_kills:.1f}</td><td>{avg_reward:.1f}</td>"
                f"<td>{avg_steps:.0f}</td><td>{avg_lat:.1f}s</td></tr>\n"
            )

        display(DHTML(
            f"<h4>{agent_name} ({model})</h4>"
            "<table style='border-collapse:collapse; font-family:monospace'>"
            "<tr style='background:#333; color:#fff'>"
            "<th style='padding:4px 12px'>Episode</th>"
            "<th style='padding:4px 12px'>Kills</th>"
            "<th style='padding:4px 12px'>Reward</th>"
            "<th style='padding:4px 12px'>Steps</th>"
            "<th style='padding:4px 12px'>Avg Latency</th></tr>\n"
            + rows +
            "</table>"
        ))

elif _dm_mode_used == "Benchmark":
    # DM Benchmark: per-agent table with episodes
    for agent_result in _dm_results:
        agent_name = agent_result["agent"]
        model = agent_result["model"]
        episodes = agent_result.get("episodes", [])

        rows = ""
        for ep in episodes:
            frags = ep.get("frags", 0)
            deaths = ep.get("deaths", 0)
            kd = frags / max(deaths, 1)
            lat = ep.get("avg_latency", 0)
            rows += (
                f"<tr><td>{ep.get('episode', '?')}</td>"
                f"<td>{frags:.0f}</td><td>{deaths:.0f}</td>"
                f"<td>{kd:.2f}</td><td>{lat:.1f}s</td></tr>\n"
            )

        if episodes:
            n = len(episodes)
            avg_frags = sum(e.get("frags", 0) for e in episodes) / n
            avg_deaths = sum(e.get("deaths", 0) for e in episodes) / n
            avg_kd = avg_frags / max(avg_deaths, 1)
            avg_lat = sum(e.get("avg_latency", 0) for e in episodes) / n
            rows += (
                f"<tr style='font-weight:bold; border-top:2px solid #888'>"
                f"<td>AVG</td><td>{avg_frags:.1f}</td><td>{avg_deaths:.1f}</td>"
                f"<td>{avg_kd:.2f}</td><td>{avg_lat:.1f}s</td></tr>\n"
            )

        display(DHTML(
            f"<h4>{agent_name} ({model})</h4>"
            "<table style='border-collapse:collapse; font-family:monospace'>"
            "<tr style='background:#333; color:#fff'>"
            "<th style='padding:4px 12px'>Episode</th>"
            "<th style='padding:4px 12px'>Frags</th>"
            "<th style='padding:4px 12px'>Deaths</th>"
            "<th style='padding:4px 12px'>K/D</th>"
            "<th style='padding:4px 12px'>Avg Latency</th></tr>\n"
            + rows +
            "</table>"
        ))

else:
    # Arena: single scoreboard sorted by frags
    arena_rows = []
    for r in _dm_results:
        result = r.get("result", {})
        arena_rows.append({
            "agent": r["agent"],
            "frags": result.get("frags", 0),
            "deaths": result.get("deaths", 0),
        })
    arena_rows.sort(key=lambda x: x["frags"], reverse=True)

    rows = ""
    for i, r in enumerate(arena_rows, 1):
        kd = r["frags"] / max(r["deaths"], 1)
        rows += (
            f"<tr><td>{i}</td><td>{r['agent']}</td>"
            f"<td>{r['frags']:.0f}</td><td>{r['deaths']:.0f}</td>"
            f"<td>{kd:.2f}</td></tr>\n"
        )

    display(DHTML(
        "<h4>Arena Scoreboard</h4>"
        "<table style='border-collapse:collapse; font-family:monospace'>"
        "<tr style='background:#333; color:#fff'>"
        "<th style='padding:4px 12px'>#</th>"
        "<th style='padding:4px 12px'>Player</th>"
        "<th style='padding:4px 12px'>Frags</th>"
        "<th style='padding:4px 12px'>Deaths</th>"
        "<th style='padding:4px 12px'>K/D</th></tr>\n"
        + rows +
        "</table>"
    ))

In [ ]:
# Show recordings (GIF/MP4)

from IPython.display import Image as IPImage, Video

if RESULTS_DIR.exists():
    recordings = sorted(RESULTS_DIR.glob("*_episode_*.*"))
    if recordings:
        for rec in recordings:
            print(f"\n{rec.name} ({rec.stat().st_size / 1_000_000:.1f} MB)")
            if rec.suffix == ".gif":
                display(IPImage(filename=str(rec)))
            elif rec.suffix == ".mp4":
                display(Video(str(rec), embed=True, mimetype="video/mp4"))
    else:
        print("No recordings found. Set 'Record format' to gif or mp4 before running.")
else:
    print("No results directory yet. Run the deathmatch first.")

In [ ]:
# ZIP-package results

import zipfile

# Pack the entire workspace directory
zip_path = BASE_DIR / "doom_deathmatch_results.zip"
dirs_to_pack = [(WORKSPACE_ROOT, "workspace")] if WORKSPACE_ROOT.exists() else []

file_count = 0
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for src_dir, arc_prefix in dirs_to_pack:
        if src_dir.exists():
            for f in sorted(src_dir.rglob("*")):
                if f.is_file():
                    zf.write(f, arcname=f"{arc_prefix}/{f.relative_to(src_dir)}")
                    file_count += 1

if file_count > 0:
    size_mb = zip_path.stat().st_size / 1_000_000
    print(f"Packed {file_count} files into doom_deathmatch_results.zip ({size_mb:.1f} MB)")
    print(f"Download: {zip_path}")
else:
    zip_path.unlink(missing_ok=True)
    print("Nothing to pack — run the deathmatch first.")